# LSTM State Encoder with Reinforcement Learning

My LSTM is based on the sample transformer + reinforcement model provided by the Kaggle competition. Most of the helper functions that convert the game information to usable information for the model and vice versa are kept the same.
The original transformer was used as a game state encoder, meaning that it would take in 24 groups of information from the game and create embeddings for each group as a 128 long vector. My model takes advantage of the fact that they the inputs are in a certain sequence, rather than bagged, and seeks to learn connections over that sequence without using attention.

I'm using the pytorch LSTM which will take in the same kind of inputs, just with only 24 groups of 16 long vectors. The model will output 96 hidden features, which will be passed through a fully connected layer to recieve only one output, a score describing how good of a postition the model is currently in. Separate embedding are used for each candiate action which is explored using a Monte Carlos Tree Search, which is also a hold over from the original model. During training, the rewards are fed back to the model as follows. 1 for a win, and -1 for a loss or tie.  

### 1. Set local project and simulator paths

In [ ]:
from pathlib import Path
import platform
import sys

PROJECT_DIR = Path(r"C:\Users\joele\Documents\Github\PersoRepo\591_project")
sim_root = Path(
    r"C:\Users\joele\Documents\Github\PersoRepo\591_project"
    r"\the-pokemon-company-ptcg-ai-battle-challenge-playground"
    r"\sample_submission\sample_submission\sample_submission"
)
project_root = PROJECT_DIR

assert (sim_root / "cg" / "api.py").is_file(), f"Missing cg/api.py under {sim_root}"
library_name = "cg.dll" if platform.system() == "Windows" else "libcg.so"
assert (sim_root / "cg" / library_name).is_file(), f"Missing {library_name}"

# A loaded cg package cannot be switched just by changing sys.path.
if "cg" in sys.modules:
    loaded_package = Path(sys.modules["cg"].__file__).resolve().parent
    if loaded_package != (sim_root / "cg").resolve():
        raise RuntimeError("A different cg package is loaded. Restart the kernel, then run from section 1.")

if str(sim_root) not in sys.path:
    sys.path.insert(0, str(sim_root))
sys.dont_write_bytecode = True  # Do not create cache files in the simulator folder.
print("Local project:", PROJECT_DIR)
print("Simulator package:", sim_root / "cg")
print("Native library:", library_name)

Local project: C:\Users\joele\Documents\Github\PersoRepo\591_project
Simulator package: C:\Users\joele\Documents\Github\PersoRepo\591_project\the-pokemon-company-ptcg-ai-battle-challenge-playground\sample_submission\sample_submission\sample_submission\cg
Native library: cg.dll


### 2. Check Python environment
 CUDA is used if available

In [24]:
import importlib.util
import sys

print("Active Python:", sys.executable)
if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is missing from this notebook kernel. Select the BasicEnv "
        "interpreter: C:/Users/scept/miniconda3/envs/BasicEnv/python.exe"
    )
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU. Start with the small trial settings in section 5.")

Active Python: c:\Users\joele\miniconda3-new\envs\torch-gpu\python.exe
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Ti


### 3. Create a local run folder

Creates a timestamped folder under `PROJECT_DIR / "runs"` for checkpoints and game records. Rerunning this cell creates a new output folder. 

In [25]:
from datetime import datetime, timezone, timedelta

mst = timezone(timedelta(hours=-7), name="MST")
run_name = datetime.now(mst).strftime("run_%Y%m%d_%H")
RUN_DIR = PROJECT_DIR / "runs" / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Saving local checkpoints and games to:", RUN_DIR)

Saving local checkpoints and games to: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20


### 4. Define the model, features, search, and deck

Both players currently use the Dragapult EX deck. We chose this one since it was used by most of the top models. **Rerunning this cell creates fresh random weights.** After a kernel restart, load a saved checkpoint.

In [ ]:
import glob
import math
import os
import random
import sys

import torch
import torch.nn
import torch.nn.functional
import torch.optim

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from pathlib import Path

# sim_root and sys.path were configured in section 1.
assert (sim_root / "cg" / "api.py").is_file(), "Simulator folder not found"

from cg.api import (
    AreaType,
    Card,
    Observation,
    OptionType,
    PlayerState,
    Pokemon,
    SearchState,
    SelectContext,
    all_attack,
    all_card_data,
    search_begin,
    search_end,
    search_step,
    to_observation_class,
)

cards = all_card_data()
print(f"Simulator loaded: {len(cards)} cards")

from cg.game import battle_start, battle_finish, battle_select

# Load all card data from the API's helper function
all_card = all_card_data()
# Create a lookup table (dictionary) to quickly access card data by its cardId
card_table = {c.cardId:c for c in all_card}
card_count = max(all_card, key=lambda c: c.cardId).cardId + 1 # Max Card ID + 1

attack_count = max(all_attack(), key=lambda a: a.attackId).attackId + 1 # Max Attack ID + 1

num_words_encoder = 24
encoder_size = max(22000, 17 * card_count + 43) # Encoder input size exceeding the vocabulary size

decoder_main_feature = 8 # Feature count of SelectContext.Main
decoder_attack_offset = 14 # First index of Attack feature
decoder_card_offset = decoder_attack_offset + attack_count # First index of Card Feature
decoder_size = decoder_card_offset + (1 + decoder_main_feature + SelectContext.RECOVER_SPECIAL_CONDITION) * card_count # Decoder input vocabulary size

SEARCH_COUNT = 10 # MCTS Search count


## ! ===============================================
# Decoder Layer of Model
class DecoderLayer(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_feedforward: int):
        super(DecoderLayer, self).__init__()

        self.attention = torch.nn.MultiheadAttention(d_model, num_heads)
        self.fc1 = torch.nn.Linear(d_model, d_feedforward)
        self.fc2 = torch.nn.Linear(d_feedforward, d_model)
        self.norm1 = torch.nn.LayerNorm(d_model)
        self.norm2 = torch.nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, encoder_out: torch.Tensor) -> torch.Tensor:
        y, _ = self.attention(x, encoder_out, encoder_out, need_weights=False)
        res = self.norm1(x + y)
        y = self.fc1(res)
        y = torch.nn.functional.relu(y)
        y = self.fc2(y)
        return self.norm2(res + y)

# LSTM Model
class LSTMModel(torch.nn.Module):
    def __init__(self,
                 embed_dim: int,
                 hidden_dim: int,
                 layer_dim: int,
                 output_dim: int,
    ):
        super(LSTMModel, self).__init__()

        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.embed_dim = embed_dim
        ## Replicating Architecture from Transformer

        ## Takes the feature inputs in their raw form (HP, energy,...) and changes
        ## them to embedding vectors according to the embed_dim set

        ## encoder_bag contains a trainable lookup table -- a table of weights, training updates them
        ## encoder_bag is for the current game state.
        ## This function is reflected in the name for the fully connect function
        ## for output v
        self.encoder_bag = torch.nn.EmbeddingBag(encoder_size, embed_dim, mode="sum")
        self.state_lstm = torch.nn.LSTM(input_size = embed_dim, hidden_size = hidden_dim, num_layers = layer_dim, batch_first = True)
        self.gamestate_fc = torch.nn.Linear(hidden_dim, out_features = output_dim)

        ## decoder_bag is for the candidate actions
        ## for output p
        self.decoder_bag = torch.nn.EmbeddingBag(decoder_size, embed_dim, mode = "sum")
        ## MLP NN for scoring the candidate actions
        self.candidates = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim + embed_dim, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, output_dim),
            torch.nn.Tanh()
        )        


## * ======== Checkpoint
    def forward(self,
                index_encoder,
                value_encoder,
                offset_encoder,
                index_decoder,
                value_decoder,
                offset_decoder
    ):
 


        ## * v is the lstm output -- board state score
        ## Features of current board state getting embedded for weights
        v = self.encoder_bag(index_encoder, offset_encoder, value_encoder)

        ##  Embeddings to observations
        v = v.reshape(-1, num_words_encoder, self.embed_dim)

        lstm_output, (hn, cn) = self.state_lstm(v)

        board_summary = hn[-1]

        v = torch.tanh(self.gamestate_fc(board_summary))

## ***======
## ????????
        ## embedding candidate actions
        p = self.decoder_bag(index_decoder, offset_decoder, value_decoder)

        ## Organize candidates by observation.
        batch_size = board_summary.size(0)
        p = p.reshape(batch_size, -1, self.embed_dim )

        ## Provide the same board summary to every candidate.
        board_to_actions = board_summary.unsqueeze(1).expand(-1, p.size(1), -1)

        ## Combine board and action information.
        combined = torch.cat((board_to_actions, p), dim=-1)

        ## Predict one score per candidate.
        p = self.candidates(combined).squeeze(-1)

        return (v, p)
## ! ===============================================

# torch.nn.EmbeddingBag input
class SparseVector:
    index: list[int]
    value: list[float]
    offset: list[int]
    pos: int

    def __init__(self):
        self.index = []
        self.value = []
        self.offset = []
        self.pos = 0

    def add(self, index: int, value: float | int | bool):
        value = float(value)
        if value != 0.0:
            self.index.append(self.pos + index)
            self.value.append(value)

    def add_pos(self, pos: int):
        self.pos += pos

    def add_single(self, value: float | int | bool):
        value = float(value)
        if value != 0.0:
            self.index.append(self.pos)
            self.value.append(value)
        self.pos += 1

    def word_start(self):
        self.offset.append(len(self.index))

# Add encoder card feature
def add_card(sv: SparseVector, card: Card | Pokemon | None):
    if card != None:
        sv.add(card.id, 1)
    sv.add_pos(card_count)

# Add encoder cards feature
def add_cards(sv: SparseVector, cards: list[Card] | None, value: float):
    if cards != None:
        for card in cards:
            sv.add(card.id, value)
    sv.add_pos(card_count)

# Add encoder Pokémon feature
def add_pokemon(sv: SparseVector, poke: Pokemon | None):
    if poke == None:
        sv.add_single(1)
        sv.add_pos(1 + 3 * card_count)
    else:
        sv.add_single(0)
        sv.add_single(poke.hp / 400)
        add_card(sv, poke)
        add_cards(sv, poke.tools, 1.0)
        add_cards(sv, poke.energyCards, 0.5)

# Add encoder player feature
def add_player(sv: SparseVector, ps: PlayerState):
    sv.add_single(ps.deckCount / 60)
    sv.add_single(len(ps.discard) / 60)
    sv.add_single(ps.handCount / 8)
    sv.add_single(len(ps.bench) / 5)
    sv.add(len(ps.prize), 1)
    sv.add_pos(7)

    sv.add_single(ps.poisoned)
    sv.add_single(ps.burned)
    sv.add_single(ps.asleep)
    sv.add_single(ps.paralyzed)
    sv.add_single(ps.confused)

    add_cards(sv, ps.discard, 0.25)

def get_encoder_input(obs: Observation, your_deck: list[int]) -> SparseVector:
    your_index = obs.current.yourIndex
    state = obs.current

    sv = SparseVector()
    for i in range(2):
        ps = state.players[i ^ your_index]
        for j in range(8): # For bench
            sv.word_start()
            pos = sv.pos
            if j < len(ps.bench):
                add_pokemon(sv, ps.bench[j])
            else:
                add_pokemon(sv, None)
            if j != 7:  # Not last
                sv.pos = pos  # Return to the previous position

    for i in range(2):
        ps = state.players[i ^ your_index]
        sv.word_start()
        if 0 < len(ps.active):
            add_pokemon(sv, ps.active[0])
        else:
            add_pokemon(sv, None)

    for i in range(2):
        ps = state.players[i ^ your_index]
        sv.word_start()
        add_player(sv, ps)

    sv.word_start()
    add_cards(sv, state.players[your_index].hand, 0.25)

    sv.word_start()
    for id in your_deck:
        sv.add(id, 0.25)
    sv.add_pos(card_count)

    sv.word_start()
    add_cards(sv, state.stadium, 1.0)

    sv.word_start()
    sv.add_single(1)
    sv.add_single(state.turn / 10)
    sv.add_single(state.firstPlayer == your_index)
    return sv

def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

# Add decoder Main Select feature
def decoder_main(sv: SparseVector, feature_index: int, card: Card | Pokemon | None):
    if card != None:
        sv.add(decoder_card_offset + feature_index * card_count + card.id, 1)

# Add decoder Card ID feature
def decoder_card_id(sv: SparseVector, context: SelectContext, card_id: int):
    sv.add(decoder_card_offset + (decoder_main_feature + context) * card_count + card_id, 1)

# Add decoder Card feature
def decoder_card(sv: SparseVector, context: SelectContext, card: Card | Pokemon | None):
    if card != None:
        decoder_card_id(sv, context, card.id)

def get_decoder_input(obs: Observation, actions: list[list[int]]) -> SparseVector:
    sv = SparseVector()
    your_index = obs.current.yourIndex
    ps = obs.current.players[your_index]
    context = obs.select.context
    for action in actions:
        sv.word_start()

        if len(action) == 0:
            sv.add(0, 1)
            continue

        for i in action:
            o = obs.select.option[i]
            match o.type:
                case OptionType.END:
                    sv.add(1, 1)
                case OptionType.YES:
                    sv.add(2, 1)
                case OptionType.NO:
                    sv.add(3, 1)
                case OptionType.SPECIAL_CONDITION:
                    sv.add(4 + o.specialConditionType, 1)
                case OptionType.NUMBER:
                    sv.add(9 + min(o.number, 4), 1)
                case OptionType.ATTACK:
                    sv.add(decoder_attack_offset + o.attackId, 1)
                case OptionType.PLAY:
                    decoder_main(sv, 0, ps.hand[o.index])
                case OptionType.ATTACH:
                    decoder_main(sv, 1, get_card(obs, o.area, o.index, your_index))
                    decoder_main(sv, 2, get_card(obs, o.inPlayArea, o.inPlayIndex, your_index))
                case OptionType.EVOLVE:
                    decoder_main(sv, 3, get_card(obs, o.area, o.index, your_index))
                    decoder_main(sv, 4, get_card(obs, o.inPlayArea, o.inPlayIndex, your_index))
                case OptionType.ABILITY:
                    decoder_main(sv, 5, get_card(obs, o.area, o.index, your_index))
                case OptionType.DISCARD:
                    decoder_main(sv, 6, get_card(obs, o.area, o.index, your_index))
                case OptionType.RETREAT:
                    decoder_main(sv, 7, ps.active[0])
                case OptionType.CARD:
                    decoder_card(sv, context, get_card(obs, o.area, o.index, o.playerIndex))
                case OptionType.TOOL_CARD:
                    card = get_card(obs, o.area, o.index, o.playerIndex)
                    decoder_card(sv, context, card.tools[o.toolIndex])
                case OptionType.ENERGY_CARD | OptionType.ENERGY:
                    card = get_card(obs, o.area, o.index, o.playerIndex)
                    decoder_card(sv, context, card.energyCards[o.energyIndex])
                case OptionType.SKILL:
                    decoder_card_id(sv, context, o.cardId)

    return sv


## ** ==============================================================================
# Evaluate with My Model
def eval_nn(sv_enc: SparseVector, sv_dec:SparseVector, model: LSTMModel) -> tuple[float, list[float]]:
    device = next(model.parameters()).device
    value, policy = model(
        torch.tensor(sv_enc.index, dtype=torch.int32, device=device),
        torch.tensor(sv_enc.value, dtype=torch.float32, device=device),
        torch.tensor(sv_enc.offset, dtype=torch.int32, device=device),
        torch.tensor(sv_dec.index, dtype=torch.int32, device=device),
        torch.tensor(sv_dec.value, dtype=torch.float32, device=device),
        torch.tensor(sv_dec.offset, dtype=torch.int32, device=device))

    return (value.tolist()[0][0], policy.tolist()[0])

## ** ==============================================================================

# Single Training Sample
class LearnSample:
    def __init__(self, value: float, policy: list[float], sv_enc: SparseVector, sv_dec:SparseVector):
        self.value = value # Encoder output
        self.policy = policy # Decoder output
        self.sv_enc = sv_enc
        self.sv_dec = sv_dec


# MCTS Node Child
class Child:
    node: 'Node | None'
    select: list[int] # Selected option indices
    prob: float # Probability

    def __init__(self, select: list[int], prob: float):
        self.node = None
        self.select = select
        self.prob = prob


# MCTS Node
class Node:
    value: float # Self value
    total: float # Total value
    visit: int # Visit count
    parent: 'Node | None' # Parent node
    children: list[Child]
    state: SearchState # Search State of this node

    def __init__(self, parent: 'Node | None', state: SearchState):
        self.value = -2.0
        self.total = 0.0
        self.visit = 0
        self.parent = parent
        self.children = []
        self.state = state

    # Backpropagation value
    def backprop(self, value: float):
        self.total += value
        self.visit += 1
        if self.parent != None:
            self.parent.backprop(value)


def create_node(parent: Node | None,
                search_state: SearchState,
                your_index: int,
                your_deck: list[int],
                model: LSTMModel
    ) -> tuple[Node, LearnSample | None]:
    node = Node(parent, search_state)

    obs = search_state.observation
    state = obs.current
    if state.result >= 0:
        # Battle finished
        if state.result == 2:
            node.value = 0
        elif state.result == your_index:
            node.value = 1
        else:
            node.value = -1
        node.backprop(node.value)
        sample = None
    else:
        actions = []
        indices = list(range(obs.select.maxCount))
        for _ in range(64):
            actions.append(indices.copy())
            for i in range(len(indices)):
                index = len(indices) - i - 1
                if indices[index] < len(obs.select.option) - i - 1:
                    indices[index] += 1
                    for j in range(index+1, len(indices)):
                        indices[j] = indices[j - 1] + 1
                    break
            else:
                break

        sv_enc = get_encoder_input(obs, your_deck)
        sv_dec = get_decoder_input(obs, actions)
        value, policy = eval_nn(sv_enc, sv_dec, model)
        v = value
        if state.yourIndex != your_index:
            v = -v
        node.value = v
        node.backprop(v)

        sum = 0.0
        for i in range(len(policy)):
            p = math.exp(policy[i] * 10.0)
            node.children.append(Child(actions[i], p))
            sum += p
        for c in node.children:
            c.prob /= sum
        sample = LearnSample(value, policy, sv_enc, sv_dec)

    return (node, sample)

# We will perform exploration using MCTS and select actions. At the same time, we will also generate training data.
def mcts_agent(obs_dict: dict, your_deck: list[int], model: LSTMModel) -> tuple[list[int], LearnSample]:
    obs = to_observation_class(obs_dict)
    your_index = obs.current.yourIndex
    state = obs.current
    active = state.players[1 - your_index].active
    search_state = search_begin(
        obs,
        your_deck=random.sample(your_deck, state.players[your_index].deckCount), # Randomly select from deck.
        your_prize=random.sample(your_deck, len(state.players[your_index].prize)), # Randomly select from deck.
        opponent_deck=[1072] * state.players[1 - your_index].deckCount, # Fill with Snorlax (There is no deep meaning).
        opponent_prize=[1] * len(state.players[1 - your_index].prize), # Fill with Basic Energy (There is no deep meaning)
        opponent_hand=[1] * state.players[1 - your_index].handCount, # Fill with Basic Energy.
        opponent_active=[1072] if len(active) > 0 and active[0] == None else []) # Fill with Snorlax.
    root, sample = create_node(None, search_state, your_index, your_deck, model) # Create root node.

    # Search
    for _ in range(SEARCH_COUNT):
        current = root
        while True:
            value = -1e9
            c = 0.4 * math.sqrt(current.visit)
            for child in current.children:
                visit = 0
                if child.node == None:
                    v = current.total / current.visit
                else:
                    v = child.node.total / child.node.visit
                    visit = child.node.visit
                if current.state.observation.current.yourIndex != your_index:
                    v = -v
                v += c * child.prob / (1 + visit)
                if value < v:
                    value = v
                    next = child

            if next.node == None:
                search_state = search_step(current.state.searchId, next.select)
                next.node, _ = create_node(current, search_state, your_index, your_deck, model)
                break
            else:
                current = next.node
                if current.state.observation.current.result >= 0:
                    current.backprop(current.value)
                    break

    # Select the most visited node.
    max_child = None
    max_visit = -1
    min_value = 10
    for child in root.children:
        if child.node != None:
            if max_visit < child.node.visit:
                max_child = child
                max_visit = child.node.visit
            v = child.node.total / child.node.visit
            if min_value > v:
                min_value = v

    # Generate training data
    sample.value = root.total / root.visit
    for i in range(len(root.children)):
        child = root.children[i]
        v = sample.value
        if child.node == None:
            v = min_value - v - 0.03
        else:
            v = child.node.total / child.node.visit - v
        sample.policy[i] = max(-1.0, min(1.0, v))

    search_end()
    return (max_child.select, sample)


# Helper class to construct batch inputs for the neural network.
class LearnInput:
    index: list[int]
    value: list[float]
    offset: list[int]

    def __init__(self):
        self.index = []
        self.value = []
        self.offset = []

    def add(self, sv: SparseVector):
        count = len(self.index)
        self.index.extend(sv.index)
        self.value.extend(sv.value)
        for o in sv.offset:
            self.offset.append(o + count)

# Opponent for evaluation.
def random_agent(obs_dict: dict) -> list[int]:
    obs = to_observation_class(obs_dict)
    return random.sample(list(range(len(obs.select.option))), obs.select.maxCount) # Select at random.

# For displaying progress.
def progress(count: int, text: str):
    current = 0
    while True:
        percent = 100 * current // count
        sys.stderr.write(f"\r{text} {percent}%   ")
        sys.stderr.flush()
        if(current >= count):
            sys.stderr.write("\n")
            sys.stderr.flush()
            break
        yield current
        current += 1

# Dragapult Deck
sample_deck = [2,2,2,5,5,5,5,7,7,112,112,119,119,119,119,120,120,120,120,121,121,121,140,235,235,1071,1080,1086,1086,1086,1086,1097,1097,1097,1120,1120,1120,1120,1121,1121,1121,1121,1152,1152,1152,1152,1182,1182,1182,1198,1198,1198,1213,1227,1227,1227,1227,1256,1256,1260]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
## * =======================================
model = LSTMModel(16, 96, 1, 1)
## * =======================================
model = model.to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn_enc = torch.nn.HuberLoss(delta=0.2)  # Encoder loss function
loss_fn_dec = torch.nn.HuberLoss(reduction="none", delta=0.1)  # Decoder loss function
# Checkpoint directories are managed through RUN_DIR.

import cg.api

print("Simulator:", cg.api.__file__)
print("PyTorch:", torch.__version__)
print("Device:", device)
print("Cards loaded:", len(all_card))
print("Encoder capacity:", encoder_size)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Ti
Simulator loaded: 1431 cards
Simulator: C:\Users\joele\Documents\Github\PersoRepo\591_project\the-pokemon-company-ptcg-ai-battle-challenge-playground\sample_submission\sample_submission\sample_submission\cg\api.py
PyTorch: 2.11.0+cu128
Device: cuda
Cards loaded: 1431
Encoder capacity: 24387


## 5. Train through self-play

Controls: `cycles`, `eval_games`, and `training_games`. Each cycle first evaluates against random play, then collects LSTM vs itself games, then updates weights from those samples and saves a checkpoint. The MCTS/search logic, loss functions, batching, target blending, and deck are kept from the original sample model. Ties receive -1 in training. Pre-cycle evaluation is not evaluation of that cycle's final weights.

In [27]:
# The main training loop.
# the for loop controls number of cycles
# nested for loops control other parameters for a training run
# THere are # of cycles,, # of games, # of self-play games
import time

# Small laptop trial; increase after checking cycle duration.
# Large trial: 7,75,150
cycles = 7
eval_games = 75
training_games = 150

def save_checkpoint(name):
    path = RUN_DIR / f"{name}.pth"

    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_type": "LSTMModel",
        "model_config": {
            "embed_dim": model.encoder_bag.embedding_dim,
            "hidden_dim": model.state_lstm.hidden_size,
            "layer_dim": model.state_lstm.num_layers,
            "output_dim": model.gamestate_fc.out_features,
        },
        # Positional constructor arguments, retained for compatibility.
        "architecture": [
            model.encoder_bag.embedding_dim,
            model.state_lstm.hidden_size,
            model.state_lstm.num_layers,
            model.gamestate_fc.out_features,
        ],
        "encoder_size": encoder_size,
        "decoder_size": decoder_size,
        "sample_deck": list(sample_deck),
        "search_count": SEARCH_COUNT,
    }, path)

    print("Saved:", path)
    return path

# For getting a time estimate
run_start = time.perf_counter()

for counter in range(cycles):
    torch.save(model.state_dict(), RUN_DIR / f"before_cycle_{counter + 1:03d}.pth")  # Local pre-update weights.
    sample_list:list[LearnSample] = []  # List of training data samples.
    cycle_start = time.perf_counter()

    model.eval()
    with torch.inference_mode():
        # Evaluation
        results = [0, 0, 0]

        for i in progress(eval_games, "Evaluating... "):
            obs, start_data = battle_start(sample_deck, sample_deck)
            if start_data.errorPlayer >= 0:
                error = "Deck error."
                if start_data.errorType == 1:
                    error = "The deck contains invalid card ID."
                elif start_data.errorType == 2:
                    error = "You can include up to four cards with the same name in the deck, excluding basic Energy cards."
                elif start_data.errorType == 3:
                    error = "There are no Basic Pokémon in the deck."
                elif start_data.errorType == 4:
                    error = "You can include only one Ace Spec card in the deck."
                raise ValueError(error)
            your_index = i % 2
            while True:
                # Break the loop if the game has ended.
                if obs["current"]["result"] >= 0:
                    break

                if obs["current"]["yourIndex"] == your_index:
                    selected, _ = mcts_agent(obs, sample_deck, model)
                else:
                    selected = random_agent(obs)
                obs = battle_select(selected)

            battle_finish()  # Finalize the game.

            if obs["current"]["result"] == 2:  # Draw
                results[2] += 1
            elif obs["current"]["result"] == your_index:  # Win
                results[0] += 1
            else: # Lose
                results[1] += 1
        wins, losses, draws = results
        total = wins + losses + draws
        rate = 100 * wins / total if total else 0.0
        print(
            f"Before-training evaluation vs random: {wins} wins, "
            f"{losses} losses, {draws} draws | Win rate: {rate:.1f}%",
            flush=True,
        )

        # Self Play
        for _ in progress(training_games, "Training Data Collecting... "):
            obs, _ = battle_start(sample_deck, sample_deck)
            samples:list[list[LearnSample]] = [[], []]  # [Player0 samples, Player1 samples]
            while True:
                if obs["current"]["result"] >= 0:
                    break

                # The MCTS agent generates an action and a training sample.
                selected, sample = mcts_agent(obs, sample_deck, model)
                samples[obs["current"]["yourIndex"]].append(sample)
                obs = battle_select(selected)

            battle_finish()  # Finalize the game.

            # Calculate the training labels and add them to the training data list.
            for i in range(2):
                LAMBDA = 0.9
                # The final value is 1.0 for a win and -1.0 for a loss.
                # Treats ties as loss for both
                value = 1.0 if i == obs["current"]["result"] else -1.0

                # Iterate backwards from the end of the game to calculate values.
                for sample in reversed(samples[i]):
                    label = (value + sample.value) * 0.5
                    value = value * LAMBDA + sample.value * (1.0 - LAMBDA)
                    sample.value = label
                    sample_list.append(sample)

    # Train on the training data collected through self-play.
    print("Training Start.")
    model.train()
    random.shuffle(sample_list)
    BATCH_SIZE = 128
    batch_count = len(sample_list) // BATCH_SIZE
    print("Training samples:", len(sample_list))
    print("Training batches:", batch_count)
    for i in range(batch_count):
        # Prepare a batch of data.
        input_enc = LearnInput()
        input_dec = LearnInput()
        mask = []
        label_enc = []
        label_dec = []
        start = BATCH_SIZE * i
        for j in range(start, start + BATCH_SIZE):
            sample = sample_list[j]
            input_enc.add(sample.sv_enc)
            input_dec.add(sample.sv_dec)
            label_enc.append(sample.value)
            label_dec.extend(sample.policy)
            for _ in range(len(sample.policy)):
                mask.append(1.0)
            for _ in range(64 - len(sample.policy)):
                mask.append(0.0)
                label_dec.append(0.0)
                input_dec.offset.append(len(input_dec.index))

        # Convert data to PyTorch tensors.
        mask_tensor = torch.tensor(mask, dtype=torch.float32, device=device)
        mask_tensor = mask_tensor.view(BATCH_SIZE, -1)
        label_tensor_enc = torch.tensor(label_enc, dtype=torch.float32, device=device)
        label_tensor_enc = label_tensor_enc.view(BATCH_SIZE, -1)
        label_tensor_dec = torch.tensor(label_dec, dtype=torch.float32, device=device)
        label_tensor_dec = label_tensor_dec.view(BATCH_SIZE, -1)

        optimizer.zero_grad()

        # Get model predictions for the batch.
        out_enc, out_dec = model(
            torch.tensor(input_enc.index, dtype=torch.int32, device=device),
            torch.tensor(input_enc.value, dtype=torch.float32, device=device),
            torch.tensor(input_enc.offset, dtype=torch.int32, device=device),
            torch.tensor(input_dec.index, dtype=torch.int32, device=device),
            torch.tensor(input_dec.value, dtype=torch.float32, device=device),
            torch.tensor(input_dec.offset, dtype=torch.int32, device=device))

        # Calculate loss.
        loss_enc = loss_fn_enc(out_enc, label_tensor_enc)
        loss_dec = loss_fn_dec(out_dec, label_tensor_dec)
        loss_dec = loss_dec * mask_tensor
        loss_dec = loss_dec.sum() / float(BATCH_SIZE)
        loss = loss_enc + loss_dec

        # Backpropagate the loss and update model parameters.
        loss.backward()
        optimizer.step()
    print("Training Finish.")
    save_checkpoint(f"after_cycle_{counter + 1:03d}")

    cycle_seconds = time.perf_counter() - cycle_start
    elapsed = time.perf_counter() - run_start
    average_cycle = elapsed / (counter + 1)
    remaining = average_cycle * (cycles - counter - 1)

    print(
        f"\nCycle {counter + 1}/{cycles}"
        f"\nThis cycle: {cycle_seconds / 60:.1f} minutes"
        f"\nElapsed: {elapsed / 60:.1f} minutes"
        f"\nEstimated remaining: {remaining / 60:.1f} minutes",
        flush=True,
    )

Evaluating...  100%   


Before-training evaluation vs random: 46 wins, 29 losses, 0 draws | Win rate: 61.3%


Training Data Collecting...  100%   


Training Start.
Training samples: 17064
Training batches: 133
Training Finish.
Saved: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\after_cycle_001.pth

Cycle 1/7
This cycle: 6.9 minutes
Elapsed: 6.9 minutes
Estimated remaining: 41.4 minutes


Evaluating...  100%   


Before-training evaluation vs random: 63 wins, 12 losses, 0 draws | Win rate: 84.0%


... (18 outputs left out)

Evaluating...  100%   


Before-training evaluation vs random: 71 wins, 4 losses, 0 draws | Win rate: 94.7%


Training Data Collecting...  100%   


Training Start.
Training samples: 24078
Training batches: 188
Training Finish.
Saved: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\after_cycle_007.pth

Cycle 7/7
This cycle: 11.1 minutes
Elapsed: 77.9 minutes
Estimated remaining: 0.0 minutes


## 6. Save a named baseline

In [ ]:
# Store current weights
# Change if retraining for new version
BASELINE_ID = "lstm_fulltrial_v5"
baseline_path = save_checkpoint(BASELINE_ID)

model.eval()

print("Baseline saved:", baseline_path)
print("Search count:", SEARCH_COUNT)

Saved: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\lstm_fulltrial_v5.pth
Baseline saved: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\lstm_fulltrial_v5.pth
Search count: 10


## 7. Evaluate the saved baseline against random player and collect games

Set `COLLECTION_GAMES` (default: 2 for a small check; an even count balances seats). Loads a separate model from the named checkpoint, alternates seats, and gives both players the unchanged deck. 

Make sure to save the trained model under an appropriate id if you want to run games against the random choice player.


In [29]:
# Evaluate a saved LSTM + MCTS checkpoint against random play.
# This cell never trains or changes the training model/optimizer.
import gzip
import hashlib
import json
import re
import uuid

COLLECTION_GAMES = 5000  # First verify two games; increase to an even count later.
assert isinstance(COLLECTION_GAMES, int) and COLLECTION_GAMES > 0

# Load a separate model so records always correspond to the saved baseline,
# even if the training model in memory has since changed.
def load_lstm_for_evaluation(checkpoint_path):
    """Restore the actual saved LSTM, including older mislabeled checkpoints."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    state_dict = checkpoint["model_state_dict"]
    if "state_lstm.weight_hh_l0" not in state_dict:
        raise ValueError(
            "This checkpoint does not contain LSTM weights. Choose a saved "
            "LSTM baseline, not a LSTM checkpoint."
        )

    # Older saves incorrectly contained LSTM architecture metadata.
    # Infer the four actual LSTM constructor settings from their weight shapes.
    inferred_config = {
        "embed_dim": state_dict["encoder_bag.weight"].shape[1],
        "hidden_dim": state_dict["state_lstm.weight_hh_l0"].shape[1],
        "layer_dim": sum(
            re.fullmatch(r"state_lstm\.weight_ih_l\d+", key) is not None
            for key in state_dict
        ),
        "output_dim": state_dict["gamestate_fc.weight"].shape[0],
    }
    saved_config = checkpoint.get("model_config")
    if saved_config is not None and saved_config != inferred_config:
        raise ValueError("Saved model configuration does not match its LSTM weights.")
    if saved_config is None:
        print("Older LSTM checkpoint: recovered configuration from weights:", inferred_config)

    if inferred_config["output_dim"] != 1:
        raise ValueError("MCTS requires one position value and one score per candidate.")
    if state_dict["encoder_bag.weight"].shape[0] != encoder_size:
        raise ValueError("Encoder vocabulary mismatch; use the matching simulator/card data.")
    if state_dict["decoder_bag.weight"].shape[0] != decoder_size:
        raise ValueError("Decoder vocabulary mismatch; use the matching simulator/card data.")
    if checkpoint["sample_deck"] != list(sample_deck):
        raise ValueError("Baseline deck mismatch.")
    if checkpoint["search_count"] != SEARCH_COUNT:
        raise ValueError("Baseline search budget mismatch.")
    if Path(checkpoint_path).stem != BASELINE_ID:
        raise ValueError("BASELINE_ID must match the selected checkpoint filename.")

    restored_model = LSTMModel(**inferred_config).to(device)
    restored_model.load_state_dict(state_dict, strict=True)
    restored_model.eval()
    return restored_model, inferred_config


evaluation_model, evaluation_config = load_lstm_for_evaluation(baseline_path)
print("Evaluating LSTM checkpoint:", baseline_path)
print("LSTM configuration:", evaluation_config)

hasher = hashlib.sha256()
with open(baseline_path, "rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        hasher.update(block)
checkpoint_sha256 = hasher.hexdigest()

# Separate every collection run from earlier self-play or evaluation datasets.
collection_id = uuid.uuid4().hex
DATA_DIR = RUN_DIR / "baseline_games" / BASELINE_ID / "vs_random" / collection_id
DATA_DIR.mkdir(parents=True, exist_ok=False)
SUMMARY_PATH = DATA_DIR / "summary.json"
wins = losses = draws = 0
game_summaries = []

def save_collection_summary():
    completed = wins + losses + draws
    summary = {
        "collection_id": collection_id,
        "baseline_id": BASELINE_ID,
        "model_type": "LSTMModel",
        "model_config": evaluation_config,
        "checkpoint_path": str(baseline_path),
        "checkpoint_sha256": checkpoint_sha256,
        "opponent": "random",
        "search_count": SEARCH_COUNT,
        "deck": list(sample_deck),
        "requested_games": COLLECTION_GAMES,
        "completed_games": completed,
        "wins": wins,
        "losses": losses,
        "draws": draws,
        "win_rate": wins / completed if completed else None,
        "win_rate_definition": "LSTM wins / all completed games",
        "draw_outcome": -1,
        "games": game_summaries,
    }
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

save_collection_summary()
print("Collecting LSTM vs random into:", DATA_DIR)

with torch.inference_mode():
    for game_number in range(COLLECTION_GAMES):
        lstm_player = game_number % 2  # Alternate player 0 / player 1.
        game_id = uuid.uuid4().hex
        records = []

        # Both players use the existing, unchanged sample deck.
        obs, start_data = battle_start(sample_deck, sample_deck)
        if obs is None:
            raise RuntimeError(f"Battle failed: {start_data.errorType}")

        try:
            while obs["current"]["result"] < 0:
                player = obs["current"]["yourIndex"]
                is_lstm = player == lstm_player

                if is_lstm:
                    selected, _ = mcts_agent(obs, sample_deck, evaluation_model)
                else:
                    selected = random_agent(obs)

                # Serialize before stepping, preserving the acting player's view.
                records.append(json.dumps({
                    "game_id": game_id,
                    "step": len(records),
                    "player": player,
                    "agent_id": BASELINE_ID if is_lstm else "random",
                    "observation": obs,
                    "selected_action": list(selected),
                }))
                obs = battle_select(selected)

            result = obs["current"]["result"]
        finally:
            battle_finish()

        output_path = DATA_DIR / f"{game_id}.jsonl.gz"
        with gzip.open(output_path, "wt", encoding="utf-8") as f:
            for record in records:
                row = json.loads(record)
                is_lstm = row["player"] == lstm_player
                row.update({
                    "collection_id": collection_id,
                    "baseline_id": BASELINE_ID if is_lstm else "random",
                    "evaluated_baseline_id": BASELINE_ID,
                    "checkpoint_sha256": checkpoint_sha256,
                    "lstm_player": lstm_player,
                    "search_count": SEARCH_COUNT if is_lstm else 0,
                    "deck": list(sample_deck),
                    "opponent": "random" if is_lstm else BASELINE_ID,
                    "result": result,
                    # Preserve the project's convention: draw = -1 for both.
                    "outcome": 1 if result == row["player"] else -1,
                })
                f.write(json.dumps(row) + "\n")

        if result == 2:
            draws += 1
            lstm_outcome = "draw"
        elif result == lstm_player:
            wins += 1
            lstm_outcome = "win"
        else:
            losses += 1
            lstm_outcome = "loss"

        game_summaries.append({
            "game_id": game_id,
            "lstm_player": lstm_player,
            "result": result,
            "lstm_outcome": lstm_outcome,
            "decision_count": len(records),
            "file": output_path.name,
        })
        save_collection_summary()  # Persist progress after every completed game.
        completed = wins + losses + draws
        print(
            f"Game {game_number + 1}/{COLLECTION_GAMES}: "
            f"LSTM seat {lstm_player}, {lstm_outcome}; "
            f"{wins} wins, {losses} losses, {draws} draws | "
            f"Win rate: {100 * wins / completed:.1f}%",
            flush=True,
        )

print("Saved summary:", SUMMARY_PATH)

Evaluating LSTM checkpoint: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\lstm_fulltrial_v5.pth
LSTM configuration: {'embed_dim': 16, 'hidden_dim': 96, 'layer_dim': 1, 'output_dim': 1}
Game 1/5000: LSTM seat 0, win; 1 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 2/5000: LSTM seat 1, win; 2 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 3/5000: LSTM seat 0, win; 3 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 4/5000: LSTM seat 1, win; 4 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 5/5000: LSTM seat 0, win; 5 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 6/5000: LSTM seat 1, win; 6 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 7/5000: LSTM seat 0, win; 7 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 8/5000: LSTM seat 1, win; 8 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 9/5000: LSTM seat 0, win; 9 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 10/5000: LSTM seat 1, win; 10 wins, 0 losses, 0 draws | Win rate: 100.0%
Game 11/5000: LSTM seat 

## 8. Inspect a recorded decision

Checks a saved game and displays one decision, its acting agent, opponent, and outcome.

In [30]:
# Inspect one recorded decision. Random rows are explicitly labeled.
files = sorted(DATA_DIR.glob("*.jsonl.gz"))
print("Completed game files:", len(files))
if not files:
    raise RuntimeError("No completed games saved yet. Run collection first.")

with gzip.open(files[0], "rt", encoding="utf-8") as f:
    example = json.loads(next(f))

print("Acting agent:", example["agent_id"])
print("Evaluated baseline:", example["evaluated_baseline_id"])
print("Opponent:", example["opponent"])
print("Selected action:", example["selected_action"])
print("Outcome (acting player's perspective):", example["outcome"])
print("Available options:", len(example["observation"]["select"]["option"]))

Completed game files: 5000
Acting agent: lstm_fulltrial_v5
Evaluated baseline: lstm_fulltrial_v5
Opponent: random
Selected action: [1]
Outcome (acting player's perspective): 1
Available options: 2


## 9. Read saved evaluation totals

Loads the persistent summary for the current collection run. 

In [32]:
# Display persisted evaluation results for this collection run.
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
print("Run folder:", RUN_DIR)
print("Baseline saved:", Path(baseline_path).is_file())
print("Evaluation folder:", DATA_DIR)
print("Completed games:", summary["completed_games"])
print("Wins / losses / draws:", summary["wins"], summary["losses"], summary["draws"])
if summary["win_rate"] is not None:
    print(f"LSTM win rate vs random: {100 * summary['win_rate']:.1f}%")

Run folder: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20
Baseline saved: True
Evaluation folder: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\baseline_games\lstm_fulltrial_v5\vs_random\7b7d30dfca40441dabcc49f94fa5f969
Completed games: 5000
Wins / losses / draws: 4791 209 0
LSTM win rate vs random: 95.8%


## 10. Check the saved training weights


In [33]:
# Check if weights stored

from pathlib import Path
import torch

path = Path(baseline_path)

print("Checkpoint:", path)
print("File exists:", path.is_file())

if path.is_file():
    print(f"File size: {path.stat().st_size / 1024**2:.2f} MB")

    checkpoint = torch.load(
        path, map_location="cpu", weights_only=True
    )
    saved_weights = checkpoint["model_state_dict"]

    print("Weight tensors saved:", len(saved_weights))

    matches = all(
        name in saved_weights
        and torch.equal(weight.detach().cpu(), saved_weights[name])
        for name, weight in model.state_dict().items()
    )
    print("Saved weights match current model:", matches)

Checkpoint: C:\Users\joele\Documents\Github\PersoRepo\591_project\runs\run_20261121_20\lstm_fulltrial_v5.pth
File exists: True
File size: 20.38 MB
Weight tensors saved: 12
Saved weights match current model: True
